# Module 7: LLM Capabilities - Solutions

## Setup

In [ ]:
# Install and import
!pip install -q transformers torch sentencepiece sentence-transformers

from transformers import pipeline
from sentence_transformers import SentenceTransformer
import numpy as np
import time
import warnings
warnings.filterwarnings('ignore')

# GPU setup
import torch
device = 0 if torch.cuda.is_available() else -1
print(f"Using: {'GPU' if device == 0 else 'CPU'}")

## A1: Text Classification

In [ ]:
# Load classifier
classifier = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english", device=device)

# Solution
statements = [
    "The unexamined life is not worth living",
    "Knowledge is power",
    "God is dead and we have killed him",
    "Hell is other people",
    "I think, therefore I am",
    "The only true wisdom is in knowing you know nothing"
]

positive_count = 0
negative_count = 0

for statement in statements:
    result = classifier(statement)
    label = result[0]['label']
    score = result[0]['score']
    
    print(f"'{statement}' → {label} ({score:.2%})")
    
    if label == 'POSITIVE':
        positive_count += 1
    else:
        negative_count += 1

print(f"\nSummary: {positive_count} positive, {negative_count} negative")

## A2: Named Entity Recognition

In [ ]:
# Load NER
ner = pipeline("ner", model="dslim/bert-base-NER", aggregation_strategy="simple", device=device)

# Solution
texts = [
    "Immanuel Kant was born in Königsberg in 1724",
    "Simone de Beauvoir met Jean-Paul Sartre in Paris",
    "Aristotle studied under Plato in Athens",
    "Friedrich Nietzsche lived in Germany and Switzerland", 
    "Confucius taught in ancient China during the Zhou Dynasty"
]

entity_database = {'PERSON': set(), 'LOCATION': set()}

for text in texts:
    entities = ner(text)
    print(f"\nText: {text}")
    
    for entity in entities:
        entity_type = entity['entity_group']
        entity_name = entity['word']
        
        print(f"  {entity_name}: {entity_type}")
        
        if entity_type in ['PER', 'PERSON']:
            entity_database['PERSON'].add(entity_name)
        elif entity_type in ['LOC', 'LOCATION']:
            entity_database['LOCATION'].add(entity_name)

print(f"\n📚 Database:")
print(f"People: {list(entity_database['PERSON'])}")
print(f"Places: {list(entity_database['LOCATION'])}")

## A3: Zero-Shot Classification

In [ ]:
# Load zero-shot
zero_shot = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device=device)

# Solution
philosophical_texts = [
    "What is the nature of consciousness and subjective experience?",
    "Is it ever morally permissible to lie to save a life?",
    "How can we know anything with certainty?",
    "What makes a government legitimate?",
    "Does beauty exist objectively or is it in the eye of the beholder?"
]

my_categories = [
    "philosophy of mind",
    "ethics and moral philosophy", 
    "epistemology",
    "political philosophy",
    "aesthetics",
    "metaphysics"
]

category_counts = {}

for text in philosophical_texts:
    result = zero_shot(text, candidate_labels=my_categories)
    top_category = result['labels'][0]
    confidence = result['scores'][0]
    
    print(f"Text: {text[:50]}...")
    print(f"Category: {top_category} ({confidence:.2%})\n")
    
    category_counts[top_category] = category_counts.get(top_category, 0) + 1

most_frequent = max(category_counts, key=category_counts.get)
print(f"Most frequent: {most_frequent} ({category_counts[most_frequent]} texts)")

## B1: Question Answering

In [ ]:
# Load QA
qa_model = pipeline("question-answering", model="distilbert-base-cased-distilled-squad", device=device)

# Solution
context = """
Virtue ethics is a normative ethical theory that emphasizes the character of the moral agent rather than rules or consequences. It traces back to Aristotle, who argued that virtues are character traits that enable human flourishing (eudaimonia). Unlike deontological ethics, which focuses on duties and rules, or consequentialism, which judges actions by their outcomes, virtue ethics asks what kind of person one should be. Key virtues include courage, temperance, justice, and practical wisdom (phronesis).
"""

questions = [
    "Who developed virtue ethics?",
    "What does virtue ethics emphasize?", 
    "What is eudaimonia?",
    "How does virtue ethics differ from deontological ethics?",
    "What are some key virtues mentioned?"
]

for question in questions:
    answer = qa_model(question=question, context=context)
    print(f"Q: {question}")
    print(f"A: {answer['answer']} ({answer['score']:.2%})\n")

## B2: Summarization

In [ ]:
# Load summarizer
summarizer = pipeline("summarization", model="sshleifer/distilbart-cnn-12-6", device=device)

# Solution
arguments = [
    """The trolley problem presents a moral dilemma where a runaway trolley heads toward five people tied to the tracks. You can pull a lever to divert it to a side track, killing one person instead of five. Most people say pulling the lever is morally acceptable. However, in a variant called the footbridge problem, you can save the five by pushing a large person off a bridge to stop the trolley, killing them. Most people find this morally unacceptable, even though the numbers are identical. This inconsistency reveals the complexity of moral intuitions.""",
    
    """The hard problem of consciousness concerns explaining why we have qualitative, subjective experiences. While we can explain cognitive functions like attention, memory, and behavioral responses through neuroscience, it remains mysterious why there is something it's like to be conscious. David Chalmers argues that even complete knowledge of brain functions wouldn't explain why experiences feel the way they do. This suggests consciousness involves something beyond physical processes.""",
    
    """Free will appears incompatible with determinism - the view that every event is the inevitable result of prior causes. If our actions are determined by prior states of the universe, it seems we cannot be truly responsible for them. However, compatibilists argue that free will is compatible with determinism as long as our actions flow from our own desires and reasoning, even if those are themselves determined."""
]

for i, argument in enumerate(arguments, 1):
    original_length = len(argument.split())
    summary = summarizer(argument, max_length=50, min_length=25)
    summary_text = summary[0]['summary_text']
    summary_length = len(summary_text.split())
    
    print(f"Argument {i}:")
    print(f"Original ({original_length} words): {argument[:80]}...")
    print(f"Summary ({summary_length} words): {summary_text}")
    print(f"Compression: {original_length/summary_length:.1f}:1\n")

## B3: Feature Extraction

In [ ]:
# Load embedder
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Solution
concepts = [
    "free will", "determinism", "consciousness", "moral responsibility",
    "causation", "personal identity", "knowledge", "belief", "truth", "justice"
]

embeddings = embedder.encode(concepts)

# Find most similar pair
max_similarity = 0
best_pair = None

for i in range(len(concepts)):
    for j in range(i+1, len(concepts)):
        similarity = np.dot(embeddings[i], embeddings[j])
        if similarity > max_similarity:
            max_similarity = similarity
            best_pair = (concepts[i], concepts[j])

print(f"Most similar concepts: {best_pair[0]} ↔ {best_pair[1]} ({max_similarity:.3f})")

# Find concepts similar to "consciousness"
target = "consciousness"
target_idx = concepts.index(target)
similarities = [(concepts[i], np.dot(embeddings[target_idx], embeddings[i])) 
                for i in range(len(concepts)) if i != target_idx]
similarities.sort(key=lambda x: x[1], reverse=True)

print(f"\nMost similar to '{target}':")
for concept, sim in similarities[:3]:
    print(f"  {concept}: {sim:.3f}")

## C1: Text Generation

In [ ]:
# Load generator
generator = pipeline("text-generation", model="gpt2", device=device)

# Solution
prompts = [
    "The nature of reality is",
    "Consciousness arises from",
    "The meaning of life lies in",
    "True knowledge comes from",
    "Moral obligations exist because"
]

for prompt in prompts:
    print(f"Prompt: '{prompt}'")
    
    # Conservative (T=0.7)
    conservative = generator(prompt, max_length=len(prompt.split()) + 20, temperature=0.7, 
                           num_return_sequences=1, pad_token_id=50256)
    
    # Creative (T=1.2)
    creative = generator(prompt, max_length=len(prompt.split()) + 20, temperature=1.2,
                        num_return_sequences=1, pad_token_id=50256)
    
    print(f"Conservative: {conservative[0]['generated_text']}")
    print(f"Creative: {creative[0]['generated_text']}\n")
    
    time.sleep(1)

## C2: Translation

In [ ]:
# Load translator
translator = pipeline("translation_en_to_de", model="Helsinki-NLP/opus-mt-en-de", device=device)

# Solution
philosophical_terms = [
    "being", "existence", "knowledge", "wisdom", 
    "virtue", "justice", "freedom", "consciousness"
]

print("English-German Philosophical Dictionary:")
philosophy_dictionary = {}

for term in philosophical_terms:
    translation = translator(term)
    german_term = translation[0]['translation_text']
    philosophy_dictionary[term] = german_term
    print(f"{term} → {german_term}")

# Translate famous quote
famous_quote = "I think, therefore I am"
quote_translation = translator(famous_quote)
print(f"\nFamous quote: '{famous_quote}' → '{quote_translation[0]['translation_text']}'")

## C3: Chat Models

In [ ]:
# Load chatbot
chatbot = pipeline("text-generation", model="microsoft/DialoGPT-small", device=device)

# Solution
conversation_turns = [
    "What is consciousness?",
    "Can you explain that further?", 
    "What are the main challenges in understanding this?",
    "How do different philosophers approach this question?"
]

conversation_history = []

for i, user_input in enumerate(conversation_turns, 1):
    print(f"Turn {i}:")
    print(f"Human: {user_input}")
    
    # Build context
    if conversation_history:
        context = " ".join([f"Human: {h} AI: {a}" for h, a in conversation_history])
        full_input = f"{context} Human: {user_input} AI:"
    else:
        full_input = f"Human: {user_input} AI:"
    
    response = chatbot(full_input, max_length=len(full_input.split()) + 25,
                      pad_token_id=50256, temperature=0.8)
    
    ai_response = response[0]['generated_text'].split("AI:")[-1].strip()
    if "Human:" in ai_response:
        ai_response = ai_response.split("Human:")[0].strip()
    
    print(f"AI: {ai_response}\n")
    conversation_history.append((user_input, ai_response))
    time.sleep(1)

## Final Project: Philosophy Research Assistant

In [ ]:
class PhilosophyResearchAssistant:
    def __init__(self):
        # Initialize all pipelines
        self.classifier = classifier
        self.ner = ner
        self.zero_shot = zero_shot
        self.qa_model = qa_model
        self.summarizer = summarizer
        self.embedder = embedder
        self.generator = generator
        print("🤖 Philosophy Research Assistant Ready!")
    
    def analyze_text(self, text):
        """Comprehensive analysis using all A-category tools"""
        # Sentiment
        sentiment = self.classifier(text)[0]
        
        # Entities
        entities = self.ner(text)
        people = [e['word'] for e in entities if e['entity_group'] in ['PER', 'PERSON']]
        places = [e['word'] for e in entities if e['entity_group'] in ['LOC', 'LOCATION']]
        
        # Classification
        categories = ["ethics", "metaphysics", "epistemology", "political philosophy", "aesthetics"]
        classification = self.zero_shot(text, candidate_labels=categories)
        
        return {
            'sentiment': f"{sentiment['label']} ({sentiment['score']:.2%})",
            'people': people,
            'places': places,
            'category': f"{classification['labels'][0]} ({classification['scores'][0]:.2%})"
        }
    
    def extract_insights(self, text):
        """Extract knowledge using B-category tools"""
        insights = {}
        
        # Summary
        if len(text.split()) > 50:
            summary = self.summarizer(text, max_length=80, min_length=20)
            insights['summary'] = summary[0]['summary_text']
        else:
            insights['summary'] = "Text too short for summarization"
        
        # Q&A
        questions = ["What is the main argument?", "What evidence is provided?"]
        insights['qa'] = []
        
        for q in questions:
            try:
                answer = self.qa_model(question=q, context=text)
                if answer['score'] > 0.2:
                    insights['qa'].append(f"Q: {q} A: {answer['answer']} ({answer['score']:.2%})")
            except:
                continue
        
        return insights
    
    def full_analysis(self, text):
        """Complete analysis"""
        print("📊 Full Analysis Report")
        print("=" * 40)
        
        analysis = self.analyze_text(text)
        print(f"Sentiment: {analysis['sentiment']}")
        print(f"People: {', '.join(analysis['people']) if analysis['people'] else 'None'}")
        print(f"Places: {', '.join(analysis['places']) if analysis['places'] else 'None'}")
        print(f"Category: {analysis['category']}")
        
        insights = self.extract_insights(text)
        print(f"\nSummary: {insights['summary']}")
        
        if insights['qa']:
            print("\nKey Q&A:")
            for qa in insights['qa']:
                print(f"  {qa}")
        
        return {'analysis': analysis, 'insights': insights}

# Test the assistant
assistant = PhilosophyResearchAssistant()

test_text = """
John Rawls' theory of justice as fairness presents a powerful framework for thinking about social and political arrangements. Writing in A Theory of Justice, Rawls argues that a just society is one that rational individuals would choose if they were behind a 'veil of ignorance' - not knowing their own position, talents, or circumstances in society. From this original position, Rawls contends that people would choose two principles: first, that each person should have equal basic liberties; and second, that social and economic inequalities should be arranged to benefit the least advantaged members of society.
"""

result = assistant.full_analysis(test_text)

## Summary

You've successfully implemented:
- **A1-A3**: Text understanding (classification, NER, zero-shot)
- **B1-B3**: Knowledge extraction (Q&A, summarization, similarity)
- **C1-C3**: Creative generation (text generation, translation, chat)
- **Final**: Integrated research assistant

Next: Module 8 will teach you to use these capabilities with more powerful API models!